# 04 反向传播与链式法则

上一节讲的损失函数、梯度下降，确实很像传统机器学习里的那一套。因为只要是“通过优化损失函数来学习参数”的模型，都离不开这些概念。

神经网络真正特殊的地方在这里：它有很多层，参数很多，中间计算也很多。我们不可能手工给每个参数单独推一遍梯度。

反向传播要解决的问题就是：**在一个多层复合函数里，高效算出每个参数对最终损失的影响。**

## 1. 为什么需要反向传播

假设一个简单神经网络是：

$$
\mathbf{h}=\phi(\mathbf{W}_1\mathbf{x}+\mathbf{b}_1)
$$

$$
\hat{y}=g(\mathbf{W}_2\mathbf{h}+b_2)
$$

损失函数是：

$$
\mathcal{L}=\mathcal{L}(\hat{y},y)
$$

训练时，我们需要知道每个参数该怎么改：

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{W}_1},\quad
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_1},\quad
\frac{\partial \mathcal{L}}{\partial \mathbf{W}_2},\quad
\frac{\partial \mathcal{L}}{\partial b_2}
$$

但问题是：$\mathbf{W}_1$ 并不直接连到损失函数。它先影响隐藏层 $\mathbf{h}$，隐藏层再影响预测值 $\hat{y}$，预测值再影响损失 $\mathcal{L}$。

所以我们要追踪一条影响链：

```text
W1 -> h -> y_hat -> loss
```

反向传播就是沿着这条链，从损失开始，反过来计算每个中间量、每个参数的影响。

## 2. 神经网络本质上是复合函数

先看一个普通复合函数：

$$
y=f(g(x))
$$

这里 $x$ 先经过 $g$，再经过 $f$。

神经网络也是这样。比如两层网络可以拆成：

$$
\mathbf{z}_1=\mathbf{W}_1\mathbf{x}+\mathbf{b}_1
$$

$$
\mathbf{h}=\phi(\mathbf{z}_1)
$$

$$
z_2=\mathbf{W}_2\mathbf{h}+b_2
$$

$$
\hat{y}=g(z_2)
$$

$$
\mathcal{L}=\mathcal{L}(\hat{y},y)
$$

这不是一个简单函数，而是一串函数套在一起。反向传播能工作，靠的就是链式法则。

## 3. 链式法则是什么

链式法则回答的问题是：如果 $x$ 通过中间变量 $u$ 影响 $y$，那么 $x$ 对 $y$ 的影响怎么算？

假设：

$$
u=g(x)
$$

$$
y=f(u)
$$

那么：

$$
\frac{dy}{dx}=\frac{dy}{du}\cdot\frac{du}{dx}
$$

这句话很重要：

```text
总影响 = 后半段影响 × 前半段影响
```

如果影响链更长：

$$
x \rightarrow a \rightarrow b \rightarrow c \rightarrow y
$$

那么：

$$
\frac{dy}{dx}=\frac{dy}{dc}\cdot\frac{dc}{db}\cdot\frac{db}{da}\cdot\frac{da}{dx}
$$

神经网络中的梯度，就是这样一段一段乘回来的。

## 4. 一个非常小的例子

先看一个没有神经网络符号的小例子：

$$
a=2x
$$

$$
b=a+3
$$

$$
L=b^2
$$

如果想知道 $x$ 对 $L$ 的影响，也就是：

$$
\frac{dL}{dx}
$$

可以按链式法则拆开：

$$
\frac{dL}{dx}=\frac{dL}{db}\cdot\frac{db}{da}\cdot\frac{da}{dx}
$$

分别求每一小段：

$$
\frac{dL}{db}=2b
$$

$$
\frac{db}{da}=1
$$

$$
\frac{da}{dx}=2
$$

所以：

$$
\frac{dL}{dx}=2b\cdot 1\cdot 2=4b
$$

如果 $x=1$，那么：

$$
a=2,\quad b=5,\quad L=25
$$

此时：

$$
\frac{dL}{dx}=4\times 5=20
$$

这就是反向传播的缩影：先正向算出中间值，再反向把梯度一段段传回来。

## 5. 前向传播和反向传播分别做什么

前向传播做的是：从输入出发，算出预测和损失。

```text
x -> a -> b -> L
```

在前向传播中，我们会保存中间结果，比如 $a$ 和 $b$。为什么要保存？因为反向传播求梯度时会用到它们。

反向传播做的是：从损失出发，反过来算每个变量的梯度。

```text
L -> b -> a -> x
```

所以名字叫 backpropagation，也就是把误差信息从后往前传播。

## 6. 为什么叫“误差往回传”

损失函数在最后面，它直接知道模型错了多少：

$$
\mathcal{L}(\hat{y},y)
$$

但是前面每一层并不知道自己错在哪里。第一层只知道自己算了一个隐藏表示，第二层只知道自己继续加工了这个表示。

反向传播做的事，就是把最终损失拆成每一层的责任：

- 输出层参数对损失有多大影响？
- 隐藏层参数对损失有多大影响？
- 更早的层对损失有多大影响？

这个“责任”用数学语言表示，就是梯度。

## 7. 单个神经元的反向传播

一个神经元的前向计算是：

$$
z=\mathbf{w}^{T}\mathbf{x}+b
$$

$$
a=\phi(z)
$$

损失是：

$$
\mathcal{L}=\mathcal{L}(a,y)
$$

如果我们想更新权重 $w_i$，需要计算：

$$
\frac{\partial \mathcal{L}}{\partial w_i}
$$

根据链式法则：

$$
\frac{\partial \mathcal{L}}{\partial w_i}
=
\frac{\partial \mathcal{L}}{\partial a}
\cdot
\frac{\partial a}{\partial z}
\cdot
\frac{\partial z}{\partial w_i}
$$

这三个部分分别表示：

- $\frac{\partial \mathcal{L}}{\partial a}$：输出 $a$ 变化时，损失怎么变。
- $\frac{\partial a}{\partial z}$：激活函数这一段怎么传梯度。
- $\frac{\partial z}{\partial w_i}$：权重 $w_i$ 对线性输出 $z$ 的影响。

而因为：

$$
z=w_1x_1+w_2x_2+\cdots+w_ix_i+\cdots+b
$$

所以：

$$
\frac{\partial z}{\partial w_i}=x_i
$$

这说明输入值 $x_i$ 会影响权重 $w_i$ 的更新幅度。

## 8. 偏置的梯度为什么更简单

偏置出现在这里：

$$
z=\mathbf{w}^{T}\mathbf{x}+b
$$

如果要求：

$$
\frac{\partial \mathcal{L}}{\partial b}
$$

同样用链式法则：

$$
\frac{\partial \mathcal{L}}{\partial b}
=
\frac{\partial \mathcal{L}}{\partial a}
\cdot
\frac{\partial a}{\partial z}
\cdot
\frac{\partial z}{\partial b}
$$

因为：

$$
\frac{\partial z}{\partial b}=1
$$

所以：

$$
\frac{\partial \mathcal{L}}{\partial b}
=
\frac{\partial \mathcal{L}}{\partial a}
\cdot
\frac{\partial a}{\partial z}
$$

偏置的作用是整体平移神经元的输出，因此它不像权重那样还要乘某个输入特征。

## 9. 反向传播不是新的求导规则

这一点很重要：反向传播不是一种新的数学求导规则。

它本质上就是链式法则的系统化应用。

那它为什么重要？因为神经网络可能有成千上万甚至上亿个参数。如果每个参数都从头单独推导一遍，会有大量重复计算。

反向传播的聪明之处是：

1. 前向传播时保存中间结果。
2. 反向传播时复用已经算过的局部梯度。
3. 从后往前一层层传递，避免重复计算。

所以它不是“更神秘的求导”，而是“更高效的求导组织方式”。

## 10. 局部梯度是什么

反向传播里经常说局部梯度。局部梯度就是某个小计算单元内部的导数。

比如加法：

$$
z=x+y
$$

局部梯度是：

$$
\frac{\partial z}{\partial x}=1,\quad \frac{\partial z}{\partial y}=1
$$

比如乘法：

$$
z=xy
$$

局部梯度是：

$$
\frac{\partial z}{\partial x}=y,\quad \frac{\partial z}{\partial y}=x
$$

比如 Sigmoid：

$$
a=\sigma(z)
$$

局部梯度是：

$$
\frac{\partial a}{\partial z}=\sigma(z)(1-\sigma(z))
$$

反向传播就是把后面传来的梯度，乘上当前节点的局部梯度，再继续往前传。

## 11. 为什么激活函数的导数很重要

前面我们花了很多时间讲 Sigmoid、Tanh、ReLU 的导数，现在原因就出现了。

如果某层有：

$$
a=\phi(z)
$$

反向传播必须经过：

$$
\frac{\partial a}{\partial z}=\phi'(z)
$$

如果 $\phi'(z)$ 很小，梯度传过这一层就会变小。

Sigmoid 在两端饱和时：

$$
\sigma'(z)\approx 0
$$

Tanh 在两端饱和时：

$$
\tanh'(z)\approx 0
$$

ReLU 在正半轴：

$$
\operatorname{ReLU}'(z)=1
$$

所以 ReLU 更适合深层网络，不是因为它公式简单这么浅，而是因为它在很多区域更利于梯度传播。

## 12. 反向传播和参数更新的关系

反向传播只负责算梯度。

梯度下降负责用梯度更新参数。

两者关系是：

```text
反向传播：算出梯度
梯度下降：使用梯度更新参数
```

数学上，反向传播给出：

$$
\nabla_{\theta}\mathcal{L}
$$

梯度下降执行：

$$
\theta \leftarrow \theta-\eta\nabla_{\theta}\mathcal{L}
$$

所以不要把二者混在一起。一个是算，一个是改。

## 13. 本节总结

这一节的逻辑链是：

```text
神经网络是多层复合函数
-> 参数通过很多中间变量影响损失
-> 链式法则可以计算这种间接影响
-> 反向传播就是链式法则的高效组织方式
-> 前向传播保存中间结果
-> 反向传播从损失开始，一层层计算梯度
-> 梯度下降再用这些梯度更新参数
```

先记住两句话：

1. 反向传播不是新的数学规则，它就是链式法则。
2. 反向传播负责算梯度，梯度下降负责改参数。

下一节可以继续讲训练神经网络的完整流程：前向传播、损失计算、反向传播、参数更新、epoch、batch、过拟合和验证集。